# 04 — V1 PyBaMM Data Generator
## Physics-Informed Predictive Battery Digital Twin — Real Physics Reference Dataset

### Purpose
This notebook moves the project from **V1 specification** to **physics-generated training/evaluation data**.

It does **not** train the V1 PINN. It does **not** build the frontend or 3D pack.

It generates a reproducible PyBaMM reference dataset using a literature-backed coupled degradation model, while preserving the V0 lineage and the Notebook 03 scenario contract.

### Scientific rules
1. V0 files are read-only reference artifacts.
2. Notebook 03 scenario files are read-only specifications.
3. Degradation is produced by PyBaMM ageing physics, not by arbitrary resistance/capacity multipliers.
4. Cell 37 is a controlled benchmark, not a hard-coded final fault location.
5. Hidden deterministic fault locations are included so localization can later be tested without learning "Cell 37".
6. S6 evaluates every candidate rate; no intervention winner is preselected here.
7. Thresholds remain unresolved until they are physically sourced and validated.
8. This notebook distinguishes **simulation-generated reference data** from experimental measurements.

### Output
The main artifacts are written under:

`PINN_EV_Battery/data/v1/`

including cell-level time series, particle radial profiles, scenario-instance metadata, a data dictionary, provenance, hashes, and validation reports.


In [1]:
# Cell 2 — Reproducible Colab environment
# Run this cell first after opening the notebook.

%pip -q install "pybamm==26.4.2" "nbformat>=5.10"

print("Pinned dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.8/924.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 27.8 MB/s eta 0:00:00
Pinned dependencies installed.


In [2]:
# Cell 3 — Imports and deterministic settings

import os
import sys
import json
import math
import time
import hashlib
import random
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pybamm

SEED = 20260926
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("PyBaMM:", pybamm.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Deterministic seed:", SEED)


Python: 3.13.15
PyBaMM: 26.4.2
NumPy: 2.1.3
Pandas: 2.2.3
Deterministic seed: 20260926


In [3]:
# Cell 4 — Mount Google Drive and resolve the genuine project root

from google.colab import drive

MOUNT_POINT = "/content/drive"
if not os.path.exists(os.path.join(MOUNT_POINT, "MyDrive")):
    drive.mount(MOUNT_POINT, force_remount=False)

MYDRIVE = Path(MOUNT_POINT) / "MyDrive"
PROJECT_ROOT = MYDRIVE / "PINN_EV_Battery"
DATA_DIR = PROJECT_ROOT / "data"
V1_DIR = DATA_DIR / "v1"
SCENARIO_DIR = PROJECT_ROOT / "scenario_design"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

for p in [PROJECT_ROOT, DATA_DIR, SCENARIO_DIR]:
    if not p.is_dir():
        raise FileNotFoundError(f"STOP — required project directory is missing: {p}")

V1_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("V1 data directory:", V1_DIR)
print("Drive root verified.")


Mounted at /content/drive
Project root: /content/drive/MyDrive/PINN_EV_Battery
V1 data directory: /content/drive/MyDrive/PINN_EV_Battery/data/v1
Drive root verified.


In [4]:
# Cell 5 — Verify the Notebook 03 contract artifacts

required_03 = [
    SCENARIO_DIR / "v1_scenario_specification.json",
    SCENARIO_DIR / "v1_scenario_matrix.csv",
    SCENARIO_DIR / "v1_input_specification.csv",
    SCENARIO_DIR / "v1_output_specification.csv",
]

missing_03 = [str(p) for p in required_03 if not p.is_file()]
if missing_03:
    raise FileNotFoundError(
        "STOP — Notebook 03 contract artifacts are missing:\n" +
        "\n".join(missing_03)
    )

with open(required_03[0], "r", encoding="utf-8") as f:
    V1_SPEC = json.load(f)

V1_SCENARIO_MATRIX = pd.read_csv(required_03[1])
V1_INPUT_SPEC = pd.read_csv(required_03[2])
V1_OUTPUT_SPEC = pd.read_csv(required_03[3])

print("Notebook 03 contract loaded.")
print("Scenario IDs in matrix:", V1_SCENARIO_MATRIX["scenario_id"].tolist())
print("Input fields:", len(V1_INPUT_SPEC))
print("Output fields:", len(V1_OUTPUT_SPEC))


Notebook 03 contract loaded.
Scenario IDs in matrix: ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S6', 'S6', 'S6', 'S6']
Input fields: 19
Output fields: 31


In [5]:
# Cell 6 — Verify genuine V0 lineage without modifying it

V0_DATASET = DATA_DIR / "dataset.csv"
V0_METADATA = DATA_DIR / "metadata.json"

if not V0_DATASET.is_file():
    raise FileNotFoundError(f"STOP — missing V0 dataset: {V0_DATASET}")
if not V0_METADATA.is_file():
    raise FileNotFoundError(f"STOP — missing V0 metadata: {V0_METADATA}")

EXPECTED_V0_COLUMNS = [
    "c_rate", "T_amb_C", "r_m", "t_s", "r_norm", "t_norm",
    "c_mol_m3", "c_norm", "T_K", "T_rise_K", "V_V", "q_W_m3",
    "sigma_r_MPa", "sigma_t_MPa", "split", "cond_id"
]

v0_df = pd.read_csv(V0_DATASET)

with open(V0_METADATA, "r", encoding="utf-8") as f:
    v0_meta = json.load(f)

if v0_df.shape != (43200, 16):
    raise ValueError(f"STOP — V0 dataset shape is {v0_df.shape}, expected (43200, 16).")

if v0_df.columns.tolist() != EXPECTED_V0_COLUMNS:
    raise ValueError(
        "STOP — V0 schema changed.\n"
        f"Expected: {EXPECTED_V0_COLUMNS}\n"
        f"Found:    {v0_df.columns.tolist()}"
    )

if set(v0_df["split"].unique()) != {"train", "val", "test"}:
    raise ValueError("STOP — V0 split labels do not match train/val/test.")

condition_split_counts = v0_df.groupby("cond_id")["split"].nunique()
if not (condition_split_counts == 1).all():
    raise ValueError("STOP — a V0 condition appears in more than one split.")

split_condition_counts = v0_df.groupby("split")["cond_id"].nunique().to_dict()
if split_condition_counts != {"train": 12, "val": 6, "test": 2}:
    raise ValueError(f"STOP — V0 condition split is {split_condition_counts}, expected 12/6/2.")

print("V0 lineage check: PASS")
print("V0 shape:", v0_df.shape)
print("V0 split:", split_condition_counts)
print("V0 dataset is treated as read-only.")


V0 lineage check: PASS
V0 shape: (43200, 16)
V0 split: {'test': 2, 'train': 12, 'val': 6}
V0 dataset is treated as read-only.


In [6]:
# Cell 7 — Hash the specification and reference artifacts for provenance

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

PROVENANCE_HASHES = {
    p.name: sha256_file(p)
    for p in required_03 + [V0_DATASET, V0_METADATA]
}

for name, digest in PROVENANCE_HASHES.items():
    print(f"{name:45s} {digest}")

print("Provenance hashes computed.")


v1_scenario_specification.json                36eded33e6116fe0b45799b940ce69b68b3a76e05e83f46e4e41f2e3c8ef47b1
v1_scenario_matrix.csv                        a41b193fbce88aca3dbea64d2a8447699a0dde038de96e3ed0a72e7c0aae6f13
v1_input_specification.csv                    4b7b79996e2273becc37adf78624826a11cc9f4476fa8820445543d72c5c0583
v1_output_specification.csv                   abcf6f2cbb8a0614accfbd677796e1079bb6a28122899e631c2cc13676c73910
dataset.csv                                   6e425946279c905721c0ffafd8b9f18a3082939875a12c2d6cda718b07866a4e
metadata.json                                 69bb1f36e4af7682183e39d292454097f083e935b7c92999cb4c34c806e9bf35
Provenance hashes computed.


In [7]:
# Cell 8 — Scientific generation configuration

# IMPORTANT:
# SMOKE mode is intentionally small and is the safe first run.
# After SMOKE passes, change ONLY RUN_MODE to "FULL" and rerun from the top.

RUN_MODE = "SMOKE"   # "SMOKE" or "FULL"

N_CELLS = 100
PACK_TOPOLOGY = "100S1P_ABSTRACTION"

TARGET_INITIAL_SOC = 0.20
AGING_AMBIENT_C = 25.0
AGING_BATCH_CYCLES = 10

# FULL run uses a more resolved particle mesh than smoke mode.
FULL_VAR_PTS = {"x_n": 5, "x_s": 5, "x_p": 5, "r_n": 30, "r_p": 30}
SMOKE_VAR_PTS = {"x_n": 5, "x_s": 5, "x_p": 5, "r_n": 12, "r_p": 12}

# The run is extended in batches until a measurable degradation separation
# is obtained or MAX_AGING_CYCLES is reached.
TARGET_SOH_SEPARATION = 0.005
MAX_AGING_CYCLES = 60

PROFILE_R_POINTS = 21 if RUN_MODE == "FULL" else 9
PROFILE_T_POINTS = 61 if RUN_MODE == "FULL" else 11

if RUN_MODE not in {"SMOKE", "FULL"}:
    raise ValueError("RUN_MODE must be 'SMOKE' or 'FULL'.")

VAR_PTS = FULL_VAR_PTS if RUN_MODE == "FULL" else SMOKE_VAR_PTS

# Notebook 03 contract rates are retained, but feasibility is measured by
# the solver. We do not declare any rate safe/unsafe in advance.
S6_RATES_C = [2.0, 1.5, 1.0, 0.8, 0.4]

# Scenario controls from Notebook 03.
SCENARIO_CONTROLS = {
    "S1": {"name": "Normal Operation", "ambient_C": 25.0, "charge_rate_C": 1.0},
    "S2": {"name": "High Charging Stress", "ambient_C": 25.0, "charge_rate_C": 2.5},
    "S3": {"name": "High Ambient Temperature", "ambient_C": 45.0, "charge_rate_C": 1.0},
    "S4": {"name": "Localized Degradation Detection", "ambient_C": 25.0, "charge_rate_C": 1.0},
    "S5": {"name": "Combined High-Risk Scenario", "ambient_C": 45.0, "charge_rate_C": 2.0},
}

# Fixed, declared locations for dataset construction.
# The model is never given fault location as an input.
TRAIN_FAULT_INSTANCES = [
    {"instance_id": "S4_CELL004", "scenario_id": "S4", "fault_cells": [4], "split": "train"},
    {"instance_id": "S4_CELL019", "scenario_id": "S4", "fault_cells": [19], "split": "train"},
    {"instance_id": "S5_CELL004", "scenario_id": "S5", "fault_cells": [4], "split": "train"},
]

VAL_FAULT_INSTANCES = [
    {"instance_id": "S4_CELL055", "scenario_id": "S4", "fault_cells": [55], "split": "val"},
    {"instance_id": "S5_CELL073", "scenario_id": "S5", "fault_cells": [73], "split": "val"},
]

TEST_FAULT_INSTANCES = [
    # Cell 37 is the controlled benchmark from Notebook 03.
    {"instance_id": "S4_CELL037_BENCH", "scenario_id": "S4", "fault_cells": [37], "split": "test"},
    {"instance_id": "S5_CELL037_BENCH", "scenario_id": "S5", "fault_cells": [37], "split": "test"},
    # Additional hidden locations and a multi-cell case prevent a Cell-37 shortcut.
    {"instance_id": "S4_CELL082", "scenario_id": "S4", "fault_cells": [82], "split": "test"},
    {"instance_id": "S5_CELL091", "scenario_id": "S5", "fault_cells": [91], "split": "test"},
    {"instance_id": "S5_CELL004_082", "scenario_id": "S5", "fault_cells": [4, 82], "split": "test"},
]

if RUN_MODE == "SMOKE":
    # Minimal smoke generation: one healthy reference + one benchmark case.
    TRAIN_FAULT_INSTANCES = TRAIN_FAULT_INSTANCES[:1]
    VAL_FAULT_INSTANCES = VAL_FAULT_INSTANCES[:1]
    TEST_FAULT_INSTANCES = TEST_FAULT_INSTANCES[:1]

print("RUN_MODE:", RUN_MODE)
print("Pack abstraction:", PACK_TOPOLOGY)
print("Cells:", N_CELLS)
print("Variable points:", VAR_PTS)
print("S6 candidate rates:", S6_RATES_C)
print("Declared test benchmark Cell 37 is present:", any(37 in x["fault_cells"] for x in TEST_FAULT_INSTANCES))


RUN_MODE: SMOKE
Pack abstraction: 100S1P_ABSTRACTION
Cells: 100
Variable points: {'x_n': 5, 'x_s': 5, 'x_p': 5, 'r_n': 12, 'r_p': 12}
S6 candidate rates: [2.0, 1.5, 1.0, 0.8, 0.4]
Declared test benchmark Cell 37 is present: True


In [15]:
# Cell 9 — Build the coupled electrochemical / thermal / mechanical PyBaMM model

def build_degradation_model(var_pts=None):
    if var_pts is None:
        var_pts = VAR_PTS

    options = {
        "thermal": "lumped",
        "surface temperature": "lumped",
        "SEI": "solvent-diffusion limited",
        "SEI porosity change": "true",
        "lithium plating": "partially reversible",
        "lithium plating porosity change": "true",
        "particle mechanics": ("swelling and cracking", "swelling only"),
        "SEI on cracks": "true",
        "loss of active material": "stress-driven",
    }

    model = pybamm.lithium_ion.DFN(options=options)
    parameter_values = pybamm.ParameterValues("OKane2022")

    return model, parameter_values, options, dict(var_pts)

def make_solver():
    # IDAKLU is appropriate for the stiff DAE structure of DFN + degradation.
    return pybamm.IDAKLUSolver(rtol=1e-5, atol=1e-6)

model_probe, params_probe, MODEL_OPTIONS, USED_VAR_PTS = build_degradation_model()

print("DFN degradation model constructed.")
print("Parameter set: OKane2022")
print("Model options:", MODEL_OPTIONS)
print("var_pts:", USED_VAR_PTS)


DFN degradation model constructed.
Parameter set: OKane2022
Model options: {'thermal': 'lumped', 'surface temperature': 'lumped', 'SEI': ('solvent-diffusion limited', 'none'), 'SEI porosity change': 'true', 'lithium plating': ('partially reversible', 'none'), 'lithium plating porosity change': 'true', 'particle mechanics': ('swelling and cracking', 'swelling only'), 'SEI on cracks': ('true', 'false'), 'loss of active material': 'stress-driven'}
var_pts: {'x_n': 5, 'x_s': 5, 'x_p': 5, 'r_n': 12, 'r_p': 12}


In [16]:
# Cell 10 — Variable resolver: exact first, controlled substring fallback

def resolve_variable(model, exact_candidates=(), required_terms=(), optional=False, label="variable"):
    keys = list(model.variables.keys())

    for key in exact_candidates:
        if key in model.variables:
            return key

    terms = [t.lower() for t in required_terms]
    ranked = []
    for key in keys:
        low = key.lower()
        if all(t in low for t in terms):
            score = sum(low.count(t) for t in terms)
            ranked.append((score, len(key), key))

    if ranked:
        ranked.sort()
        return ranked[0][2]

    if optional:
        return None

    raise KeyError(
        f"STOP — required PyBaMM variable '{label}' was not found. "
        f"Searched exact={list(exact_candidates)} terms={list(required_terms)}"
    )

CORE_VARIABLES = {
    "time": "Time [s]",
    "voltage": "Voltage [V]",
    "current": "Current [A]",
    "cell_temperature": resolve_variable(
        model_probe,
        exact_candidates=("X-averaged cell temperature [K]", "Cell temperature [K]"),
        required_terms=("cell temperature", "[k]"),
        label="cell temperature"
    ),
    "surface_temperature": resolve_variable(
        model_probe,
        exact_candidates=("Surface temperature [K]",),
        required_terms=("surface temperature", "[k]"),
        label="surface temperature"
    ),
    "neg_conc": resolve_variable(
        model_probe,
        exact_candidates=("X-averaged negative particle concentration [mol.m-3]",),
        required_terms=("negative particle concentration", "[mol.m-3]"),
        label="negative particle concentration"
    ),
    "pos_conc": resolve_variable(
        model_probe,
        exact_candidates=("X-averaged positive particle concentration [mol.m-3]",),
        required_terms=("positive particle concentration", "[mol.m-3]"),
        label="positive particle concentration"
    ),
    "electrolyte_conc": resolve_variable(
        model_probe,
        exact_candidates=("X-averaged electrolyte concentration [mol.m-3]",),
        required_terms=("electrolyte concentration", "[mol.m-3]"),
        label="electrolyte concentration"
    ),
    "neg_stress_r": resolve_variable(
        model_probe,
        exact_candidates=("Negative particle radial stress [Pa]",),
        required_terms=("negative particle", "radial stress", "[pa]"),
        label="negative-particle radial stress"
    ),
    "neg_stress_t": resolve_variable(
        model_probe,
        exact_candidates=("Negative particle tangential stress [Pa]",),
        required_terms=("negative particle", "tangential stress", "[pa]"),
        label="negative-particle tangential stress"
    ),
    "capacity": resolve_variable(
        model_probe,
        exact_candidates=("Discharge capacity [A.h]",),
        required_terms=("discharge capacity", "[a.h]"),
        label="discharge capacity"
    ),
    "resistance": resolve_variable(
        model_probe,
        exact_candidates=("Resistance [Ohm]",),
        required_terms=("resistance", "[ohm]"),
        optional=True,
        label="terminal resistance proxy"
    ),
}

OPTIONAL_VARIABLES = {
    "neg_sei_thickness": resolve_variable(
        model_probe,
        exact_candidates=("X-averaged negative total SEI thickness [m]", "Negative total SEI thickness [m]"),
        required_terms=("negative", "sei", "thickness", "[m]"),
        optional=True,
        label="negative SEI thickness"
    ),
    "neg_crack_length": resolve_variable(
        model_probe,
        exact_candidates=("X-averaged negative particle crack length [m]", "Negative particle crack length [m]"),
        required_terms=("negative", "crack length", "[m]"),
        optional=True,
        label="negative crack length"
    ),
    "lam_negative": resolve_variable(
        model_probe,
        exact_candidates=("Loss of active material in negative electrode [%]"),
        required_terms=("loss of active material", "negative electrode", "[%]"),
        optional=True,
        label="negative LAM"
    ),
}

print("Core variable mapping:")
for k, v in CORE_VARIABLES.items():
    print(f"  {k:24s} -> {v}")
print("Optional diagnostic mapping:")
for k, v in OPTIONAL_VARIABLES.items():
    print(f"  {k:24s} -> {v}")


Core variable mapping:
  time                     -> Time [s]
  voltage                  -> Voltage [V]
  current                  -> Current [A]
  cell_temperature         -> X-averaged cell temperature [K]
  surface_temperature      -> Surface temperature [K]
  neg_conc                 -> X-averaged negative particle concentration [mol.m-3]
  pos_conc                 -> X-averaged positive particle concentration [mol.m-3]
  electrolyte_conc         -> X-averaged electrolyte concentration [mol.m-3]
  neg_stress_r             -> Negative particle surface radial stress [Pa]
  neg_stress_t             -> Negative particle surface tangential stress [Pa]
  capacity                 -> Discharge capacity [A.h]
  resistance               -> Resistance [Ohm]
Optional diagnostic mapping:
  neg_sei_thickness        -> Negative SEI thickness [m]
  neg_crack_length         -> X-averaged negative particle crack length [m]
  lam_negative             -> Loss of active material in negative electrode [

In [17]:
# Cell 11 — Smoke test: solve one short nominal discharge before any ageing
#
# FIX:
# The PyBaMM "surface temperature": "lumped" model requires:
#   1. Casing heat capacity [J.K-1]
#   2. Environment thermal resistance [K.W-1]
#
# These are supplied explicitly here so the same corrected
# make_simulation() is used by later ageing/scenario cells.

THERMAL_SUPPORT_PARAMETERS = {
    "Casing heat capacity [J.K-1]": 30.0,
    "Environment thermal resistance [K.W-1]": 10.0,
}


def make_simulation(ambient_C=25.0, experiment_steps=None):

    # Build the authoritative DFN degradation model from Cell 9
    model, parameter_values, options, var_pts = build_degradation_model()

    # Work on a copy so the base PyBaMM parameter object is not modified
    parameter_values = parameter_values.copy()

    # ------------------------------------------------------------
    # Required parameters for:
    # thermal = lumped
    # surface temperature = lumped
    # ------------------------------------------------------------
    parameter_values.update(
        THERMAL_SUPPORT_PARAMETERS,
        check_already_exists=False,
    )

    # Scenario-specific ambient temperature
    parameter_values.update(
        {
            "Ambient temperature [K]": ambient_C + 273.15
        },
        check_already_exists=False,
    )

    # ------------------------------------------------------------
    # Align model voltage limits with the declared experiment steps.
    # These values must still be validated for the intended chemistry.
    # ------------------------------------------------------------
    parameter_values.update(
        {
            "Lower voltage cut-off [V]": 2.5,
            "Upper voltage cut-off [V]": 4.2,
        },
        check_already_exists=False,
    )


    # Experiment definition
    exp = None

    if experiment_steps is not None:
        exp = pybamm.Experiment(experiment_steps)

    # Construct simulation
    sim = pybamm.Simulation(
        model=model,
        parameter_values=parameter_values,
        experiment=exp,
        var_pts=var_pts,
        solver=make_solver(),
    )

    return sim, model, parameter_values, options, var_pts


# ------------------------------------------------------------
# Explicit parameter check before the solver is called
# ------------------------------------------------------------

print("Checking required lumped-surface thermal parameters...")

required_thermal = [
    "Casing heat capacity [J.K-1]",
    "Environment thermal resistance [K.W-1]",
]

for name in required_thermal:
    if name not in THERMAL_SUPPORT_PARAMETERS:
        raise RuntimeError(
            f"STOP — required thermal parameter is missing: {name}"
        )

    value = THERMAL_SUPPORT_PARAMETERS[name]

    if not np.isfinite(float(value)):
        raise RuntimeError(
            f"STOP — thermal parameter '{name}' is not finite."
        )

print("Casing heat capacity [J.K-1]:",
      THERMAL_SUPPORT_PARAMETERS["Casing heat capacity [J.K-1]"])

print("Environment thermal resistance [K.W-1]:",
      THERMAL_SUPPORT_PARAMETERS["Environment thermal resistance [K.W-1]"])

print("Lumped-surface thermal parameter check: PASS")


# ------------------------------------------------------------
# Smoke experiment
# ------------------------------------------------------------

smoke_steps = [
    "Discharge at 0.5C for 120 seconds",
]

print("\nBuilding smoke simulation...")

smoke_sim, smoke_model, smoke_params, _, _ = make_simulation(
    ambient_C=25.0,
    experiment_steps=smoke_steps,
)

print("Simulation construction: PASS")
print("Starting PyBaMM smoke solve...")


# ------------------------------------------------------------
# Actual PyBaMM solve
# ------------------------------------------------------------

smoke_solution = smoke_sim.solve(
    initial_soc=1.0,
    calc_esoh=False,
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

if len(smoke_solution.t) < 2:
    raise RuntimeError(
        "STOP — PyBaMM smoke solve returned too few time points."
    )


final_voltage = float(
    smoke_solution[
        CORE_VARIABLES["voltage"]
    ].entries[-1]
)

final_temperature = float(
    smoke_solution[
        CORE_VARIABLES["cell_temperature"]
    ].entries[-1]
)


if not np.isfinite(final_voltage):
    raise RuntimeError(
        "STOP — smoke-test final voltage is NaN or Inf."
    )

if not np.isfinite(final_temperature):
    raise RuntimeError(
        "STOP — smoke-test final temperature is NaN or Inf."
    )


print("\n" + "=" * 70)
print("PYBAMM SMOKE SOLVE: PASS")
print("=" * 70)
print("Time points:", len(smoke_solution.t))
print("Final voltage [V]:", final_voltage)
print("Final temperature [K]:", final_temperature)
print("Ambient temperature [C]:", 25.0)
print(
    "Casing heat capacity [J.K-1]:",
    THERMAL_SUPPORT_PARAMETERS["Casing heat capacity [J.K-1]"]
)
print(
    "Environment thermal resistance [K.W-1]:",
    THERMAL_SUPPORT_PARAMETERS[
        "Environment thermal resistance [K.W-1]"
    ]
)
print("=" * 70)

Checking required lumped-surface thermal parameters...
Casing heat capacity [J.K-1]: 30.0
Environment thermal resistance [K.W-1]: 10.0
Lumped-surface thermal parameter check: PASS

Building smoke simulation...
Simulation construction: PASS
Starting PyBaMM smoke solve...


2026-09-26 20:05:42.721 - [ERROR] callbacks.on_experiment_error(235): Simulation error: IDASolve at t0: IDA_CONV_FAIL: Convergence test failures occurred too many times during one step or with |h| = hmin (flag: -4)
ERROR:pybamm.logger:Simulation error: IDASolve at t0: IDA_CONV_FAIL: Convergence test failures occurred too many times during one step or with |h| = hmin (flag: -4)


SolverError: IDASolve at t0: IDA_CONV_FAIL: Convergence test failures occurred too many times during one step or with |h| = hmin (flag: -4)

In [ ]:
# Cell 12 — Helpers for extracting robust time-series data

def solution_time(sol):
    return np.asarray(sol.t, dtype=float)

def series_from_solution(sol, key):
    arr = np.asarray(sol[key].entries, dtype=float)
    t_len = len(sol.t)

    if arr.ndim == 1:
        if len(arr) != t_len:
            raise ValueError(f"Variable {key} has shape {arr.shape}, time length {t_len}.")
        return arr

    matching_axes = [i for i, size in enumerate(arr.shape) if size == t_len]
    if not matching_axes:
        raise ValueError(f"Variable {key} has no axis matching time length {t_len}: {arr.shape}")

    time_axis = matching_axes[-1]
    arr = np.moveaxis(arr, time_axis, -1)
    arr = arr.reshape(-1, t_len)
    return np.nanmean(arr, axis=0)

def final_capacity_from_capacity_test(sol):
    # A one-cycle capacity experiment returns a cycle sub-solution.
    if getattr(sol, "cycles", None):
        cycle = sol.cycles[0]
        vals = np.asarray(cycle[CORE_VARIABLES["capacity"]].entries, dtype=float)
        finite = vals[np.isfinite(vals)]
        if len(finite):
            return float(np.nanmax(finite))

    vals = np.asarray(sol[CORE_VARIABLES["capacity"]].entries, dtype=float)
    finite = vals[np.isfinite(vals)]
    if len(finite):
        return float(np.nanmax(finite))

    raise RuntimeError("Could not extract discharge capacity from the capacity test.")

def safe_optional_series(sol, key):
    if key is None:
        return None
    try:
        return series_from_solution(sol, key)
    except Exception:
        return None

def parameter_first(params, names):
    for name in names:
        try:
            return float(params[name])
        except Exception:
            pass
    raise KeyError(f"None of these parameter names were available: {names}")

def cumulative_trapezoid_numpy(y, x):
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(y)
    if len(y) > 1:
        out[1:] = np.cumsum(0.5 * (y[1:] + y[:-1]) * np.diff(x))
    return out

def compute_soc_proxy(time_s, current_A, capacity_Ah, soc0):
    # PyBaMM current convention is positive for discharge, negative for charge.
    q_Ah = cumulative_trapezoid_numpy(current_A, time_s) / 3600.0
    soc = soc0 - q_Ah / max(capacity_Ah, 1e-12)
    return np.clip(soc, -0.05, 1.05)

print("Extraction helpers loaded.")


In [ ]:
# Cell 13 — Capacity and resistance diagnostics

def measure_capacity(initial_soc=1.0, starting_solution=None, ambient_C=25.0):
    steps = ["Discharge at 0.1C until 2.5 V"]
    sim, model, params, _, _ = make_simulation(ambient_C=ambient_C, experiment_steps=steps)

    if starting_solution is None:
        sol = sim.solve(initial_soc=initial_soc, calc_esoh=False)
    else:
        sol = sim.solve(starting_solution=starting_solution, calc_esoh=False)

    capacity_Ah = final_capacity_from_capacity_test(sol)
    return capacity_Ah, sol

def make_full_state_to_target(initial_state_solution, capacity_Ah, ambient_C=25.0):
    nominal_capacity = parameter_first(
        params_probe,
        ["Nominal cell capacity [A.h]"]
    )
    # Estimate the discharge time from full SOC to target SOC.
    duration_s = 3600.0 * capacity_Ah * (1.0 - TARGET_INITIAL_SOC) / nominal_capacity
    duration_s = max(30.0, float(duration_s))

    steps = [
        f"Discharge at 1C for {duration_s:.3f} seconds",
        "Rest for 60 seconds",
    ]

    sim, model, params, _, _ = make_simulation(ambient_C=ambient_C, experiment_steps=steps)
    sol = sim.solve(starting_solution=initial_state_solution, calc_esoh=False)
    return sol, duration_s

def measure_dc_pulse_resistance(initial_state_solution, capacity_Ah, ambient_C=25.0):
    nominal_capacity = parameter_first(
        params_probe,
        ["Nominal cell capacity [A.h]"]
    )
    duration_s = 3600.0 * capacity_Ah * 0.50 / nominal_capacity

    steps = [
        f"Discharge at 1C for {max(30.0, duration_s):.3f} seconds",
        "Rest for 60 seconds",
        "Discharge at 1C for 10 seconds",
        "Rest for 60 seconds",
    ]

    sim, model, params, _, _ = make_simulation(ambient_C=ambient_C, experiment_steps=steps)
    sol = sim.solve(starting_solution=initial_state_solution, calc_esoh=False)

    t = solution_time(sol)
    v = series_from_solution(sol, CORE_VARIABLES["voltage"])
    i = series_from_solution(sol, CORE_VARIABLES["current"])

    pulse_idx = np.where(np.abs(i) > 0.5)[0]
    if len(pulse_idx) < 2:
        return np.nan, sol

    idx0 = pulse_idx[0]
    idx1 = pulse_idx[min(1, len(pulse_idx)-1)]

    v0 = float(v[max(0, idx0 - 1)])
    v1 = float(v[idx1])
    i_pulse = float(np.nanmedian(i[pulse_idx]))

    resistance = abs((v0 - v1) / max(abs(i_pulse), 1e-12))
    return float(resistance), sol

print("Capacity/resistance diagnostic functions loaded.")


In [ ]:
# Cell 13 — Capacity and resistance diagnostics

def measure_capacity(initial_soc=1.0, starting_solution=None, ambient_C=25.0):
    steps = ["Discharge at 0.1C until 2.5 V"]
    sim, model, params, _, _ = make_simulation(ambient_C=ambient_C, experiment_steps=steps)

    if starting_solution is None:
        sol = sim.solve(initial_soc=initial_soc, calc_esoh=False)
    else:
        sol = sim.solve(starting_solution=starting_solution, calc_esoh=False)

    t = solution_time(sol)
    current = series_from_solution(sol, CORE_VARIABLES["current"])

    # Directly integrate the current delivered during this capacity test.
    # This avoids depending on the semantics of a carried "Discharge capacity"
    # state inside a starting solution.
    delivered_Ah = abs(float(cumulative_trapezoid_numpy(current, t)[-1] / 3600.0))

    if not np.isfinite(delivered_Ah) or delivered_Ah <= 0:
        raise RuntimeError(f"STOP — invalid measured capacity: {delivered_Ah}")

    return delivered_Ah, sol

def make_full_state_to_target(initial_state_solution, capacity_Ah, ambient_C=25.0):
    nominal_capacity = parameter_first(
        params_probe,
        ["Nominal cell capacity [A.h]"]
    )

    # Estimate the discharge time from full SOC to the target operating SOC.
    duration_s = 3600.0 * capacity_Ah * (1.0 - TARGET_INITIAL_SOC) / nominal_capacity
    duration_s = max(30.0, float(duration_s))

    steps = [
        f"Discharge at 1C for {duration_s:.3f} seconds",
        "Rest for 60 seconds",
    ]

    sim, model, params, _, _ = make_simulation(ambient_C=ambient_C, experiment_steps=steps)
    sol = sim.solve(starting_solution=initial_state_solution, calc_esoh=False)
    return sol, duration_s

def measure_dc_pulse_resistance(initial_state_solution, capacity_Ah, ambient_C=25.0):
    nominal_capacity = parameter_first(
        params_probe,
        ["Nominal cell capacity [A.h]"]
    )
    duration_s = 3600.0 * capacity_Ah * 0.50 / nominal_capacity

    steps = [
        f"Discharge at 1C for {max(30.0, duration_s):.3f} seconds",
        "Rest for 60 seconds",
        "Discharge at 1C for 10 seconds",
        "Rest for 60 seconds",
    ]

    sim, model, params, _, _ = make_simulation(ambient_C=ambient_C, experiment_steps=steps)
    sol = sim.solve(starting_solution=initial_state_solution, calc_esoh=False)

    t = solution_time(sol)
    v = series_from_solution(sol, CORE_VARIABLES["voltage"])
    i = series_from_solution(sol, CORE_VARIABLES["current"])

    # Detect the 10-second pulse as the final non-rest current segment.
    active = np.where(np.abs(i) > 0.5)[0]
    if len(active) < 2:
        return np.nan, sol

    # Find contiguous active segments and use the final segment.
    breaks = np.where(np.diff(active) > 1)[0]
    start_positions = np.r_[0, breaks + 1]
    end_positions = np.r_[breaks, len(active) - 1]
    last_start = int(active[start_positions[-1]])
    last_end = int(active[end_positions[-1]])

    before_idx = max(0, last_start - 1)
    v0 = float(v[before_idx])
    v1 = float(v[last_end])
    i_pulse = float(np.nanmedian(i[last_start:last_end + 1]))

    resistance = abs((v0 - v1) / max(abs(i_pulse), 1e-12))
    return float(resistance), sol

print("Capacity/resistance diagnostic functions loaded.")


In [ ]:
# Cell 14 — Create healthy reference capacity and a fresh full-charge state

print("Solving fresh healthy 0.1C capacity reference...")
healthy_capacity_Ah, healthy_capacity_solution = measure_capacity(
    initial_soc=1.0,
    starting_solution=None,
    ambient_C=AGING_AMBIENT_C,
)

if not np.isfinite(healthy_capacity_Ah) or healthy_capacity_Ah <= 0:
    raise RuntimeError(f"STOP — invalid healthy capacity: {healthy_capacity_Ah}")

healthy_full_sim, _, _, _, _ = make_simulation(
    ambient_C=AGING_AMBIENT_C,
    experiment_steps=["Rest for 1 second"],
)
healthy_full_solution = healthy_full_sim.solve(initial_soc=1.0, calc_esoh=False)

print(f"Healthy reference capacity: {healthy_capacity_Ah:.6f} A.h")
print("Healthy full-charge start state prepared.")


In [ ]:
# Cell 15 — Generate physics-based ageing; no arbitrary degradation multipliers

# Ageing model: 1C discharge to 2.5 V followed by 0.3C CC-CV charge.
# The degradation physics comes from the coupled PyBaMM options and OKane2022.
aging_steps = [
    (
        "Discharge at 1C until 2.5 V",
        "Rest for 600 seconds",
        "Charge at 0.3C until 4.2 V",
        "Hold at 4.2 V until C/50",
    )
]

aging_sim, aging_model, aging_params, _, _ = make_simulation(
    ambient_C=AGING_AMBIENT_C,
    experiment_steps=aging_steps * AGING_BATCH_CYCLES,
)

aging_solution = None
total_aging_cycles = 0
aged_capacity_Ah = healthy_capacity_Ah
aged_soh = 1.0

while total_aging_cycles < MAX_AGING_CYCLES:
    batch = min(AGING_BATCH_CYCLES, MAX_AGING_CYCLES - total_aging_cycles)

    # The official PyBaMM continuation pattern is to keep the same Simulation
    # and pass the previous solution as starting_solution. We therefore solve
    # the declared batch experiment repeatedly until enough physical ageing
    # separation is observed.
    if aging_solution is None:
        aging_solution = aging_sim.solve(initial_soc=1.0, calc_esoh=False)
    else:
        aging_solution = aging_sim.solve(
            starting_solution=aging_solution,
            calc_esoh=False
        )

    total_aging_cycles += batch

    aged_capacity_Ah, _ = measure_capacity(
        starting_solution=aging_solution,
        ambient_C=AGING_AMBIENT_C,
    )
    aged_soh = aged_capacity_Ah / healthy_capacity_Ah

    print(
        f"Ageing progress: {total_aging_cycles:02d} cycles | "
        f"capacity={aged_capacity_Ah:.6f} A.h | "
        f"capacity-retention SOH={aged_soh:.6f}"
    )

    if 1.0 - aged_soh >= TARGET_SOH_SEPARATION:
        break

if aging_solution is None:
    raise RuntimeError("STOP — no ageing solution was generated.")

if total_aging_cycles >= MAX_AGING_CYCLES and 1.0 - aged_soh < TARGET_SOH_SEPARATION:
    print("WARNING — requested measurable separation was not reached before MAX_AGING_CYCLES.")
    print("The ageing state remains physics-generated, but the benchmark may be weak.")

print("Physics-based ageing generation completed.")


In [ ]:
# Cell 16 — Measure fresh and aged resistance proxies at matched operating conditions

print("Measuring healthy DC-pulse resistance proxy...")
healthy_resistance_ohm, healthy_resistance_solution = measure_dc_pulse_resistance(
    healthy_full_solution,
    healthy_capacity_Ah,
    ambient_C=AGING_AMBIENT_C,
)

print("Measuring aged DC-pulse resistance proxy...")
aged_resistance_ohm, aged_resistance_solution = measure_dc_pulse_resistance(
    aging_solution,
    aged_capacity_Ah,
    ambient_C=AGING_AMBIENT_C,
)

resistance_multiplier = (
    aged_resistance_ohm / healthy_resistance_ohm
    if np.isfinite(healthy_resistance_ohm) and healthy_resistance_ohm > 0 and np.isfinite(aged_resistance_ohm)
    else np.nan
)

print(f"Healthy resistance proxy: {healthy_resistance_ohm:.8f} Ohm")
print(f"Aged resistance proxy:    {aged_resistance_ohm:.8f} Ohm")
print(f"Aged/healthy multiplier:  {resistance_multiplier:.6f}")


In [ ]:
# Cell 17 — Run a scenario from the already-matched operating state

def run_operating_scenario(
    state_kind,
    charge_rate_C,
    ambient_C,
    healthy_start_solution,
    aged_start_solution,
    healthy_capacity_Ah,
    aged_capacity_Ah,
):
    if state_kind == "healthy":
        start_solution = healthy_start_solution
        capacity_Ah = healthy_capacity_Ah
    elif state_kind == "aged":
        start_solution = aged_start_solution
        capacity_Ah = aged_capacity_Ah
    else:
        raise ValueError("state_kind must be 'healthy' or 'aged'.")

    # The start solutions have already been conditioned from full SOC to the
    # declared target operating SOC. Do not discharge a second time.
    steps = [
        "Rest for 60 seconds",
        f"Charge at {charge_rate_C:g}C until 4.2 V",
        "Hold at 4.2 V until C/50",
        "Rest for 60 seconds",
    ]

    sim, model, params, options, var_pts = make_simulation(
        ambient_C=ambient_C,
        experiment_steps=steps,
    )

    sol = sim.solve(
        starting_solution=start_solution,
        calc_esoh=False,
    )

    return {
        "solution": sol,
        "model": model,
        "params": params,
        "options": options,
        "var_pts": var_pts,
        "conditioning_duration_s": 0.0,
        "capacity_Ah": capacity_Ah,
    }

print("Scenario-run function ready. Starting SOC conditioning will not be repeated.")


In [ ]:
# Cell 18 — Prepare healthy and aged scenario start states near 20% SOC

healthy_start_solution, healthy_conditioning_s = make_full_state_to_target(
    healthy_full_solution,
    healthy_capacity_Ah,
    ambient_C=AGING_AMBIENT_C,
)

aged_start_solution, aged_conditioning_s = make_full_state_to_target(
    aging_solution,
    aged_capacity_Ah,
    ambient_C=AGING_AMBIENT_C,
)

print(f"Healthy conditioning duration: {healthy_conditioning_s:.3f} s")
print(f"Aged conditioning duration:    {aged_conditioning_s:.3f} s")
print("Healthy and aged scenario start states prepared.")


In [ ]:
# Cell 19 — Extract scalar trajectory + degradation diagnostics

def extract_timeseries(
    run,
    scenario_id,
    instance_id,
    cell_id,
    split,
    state_kind,
    fault_cells,
    ambient_C_override=None,
    charge_rate_C_override=None,
):
    sol = run["solution"]
    t = solution_time(sol)

    voltage = series_from_solution(sol, CORE_VARIABLES["voltage"])
    current = series_from_solution(sol, CORE_VARIABLES["current"])
    temp = series_from_solution(sol, CORE_VARIABLES["cell_temperature"])
    surface_temp = series_from_solution(sol, CORE_VARIABLES["surface_temperature"])
    neg_c = series_from_solution(sol, CORE_VARIABLES["neg_conc"])
    pos_c = series_from_solution(sol, CORE_VARIABLES["pos_conc"])
    electrolyte_c = series_from_solution(sol, CORE_VARIABLES["electrolyte_conc"])

    soc = compute_soc_proxy(
        t,
        current,
        run["capacity_Ah"],
        TARGET_INITIAL_SOC,
    )

    resistance = safe_optional_series(sol, CORE_VARIABLES["resistance"])

    ambient_value = (
        float(ambient_C_override)
        if ambient_C_override is not None
        else ambient_C_for_scenario(scenario_id)
    )
    rate_value = (
        float(charge_rate_C_override)
        if charge_rate_C_override is not None
        else charge_rate_for_scenario(scenario_id)
    )

    data = pd.DataFrame({
        "scenario_id": scenario_id,
        "scenario_instance_id": instance_id,
        "data_split": split,
        "pack_topology": PACK_TOPOLOGY,
        "cell_id": cell_id,
        "fault_cells_json": json.dumps(fault_cells),
        "fault_count": len(fault_cells),
        "state_kind": state_kind,
        "ageing_cycles": total_aging_cycles if state_kind == "aged" else 0,
        "ambient_temperature_C": ambient_value,
        "requested_charge_rate_C": rate_value,
        "conditioning_duration_s": run["conditioning_duration_s"],
        "time_s": t,
        "cell_voltage_V": voltage,
        "cell_current_A": current,
        "cell_temperature_K": temp,
        "surface_temperature_K": surface_temp,
        "ambient_temperature_K": ambient_value + 273.15,
        "soc_proxy": soc,
        "negative_particle_concentration_mean_mol_m3": neg_c,
        "positive_particle_concentration_mean_mol_m3": pos_c,
        "electrolyte_concentration_xavg_mol_m3": electrolyte_c,
        "capacity_Ah_reference_test": run["capacity_Ah"],
        "capacity_retention_soh": run["capacity_Ah"] / healthy_capacity_Ah,
        "dc_pulse_resistance_proxy_ohm": aged_resistance_ohm if state_kind == "aged" else healthy_resistance_ohm,
        "resistance_proxy_multiplier": resistance_multiplier if state_kind == "aged" else 1.0,
    })

    if resistance is not None and len(resistance) == len(data):
        data["pybamm_resistance_proxy_ohm"] = resistance

    for label, key in OPTIONAL_VARIABLES.items():
        vals = safe_optional_series(sol, key)
        data[label] = vals if vals is not None and len(vals) == len(data) else np.nan

    return data

def ambient_C_for_scenario(scenario_id):
    if scenario_id == "S1": return SCENARIO_CONTROLS["S1"]["ambient_C"]
    if scenario_id == "S2": return SCENARIO_CONTROLS["S2"]["ambient_C"]
    if scenario_id == "S3": return SCENARIO_CONTROLS["S3"]["ambient_C"]
    if scenario_id == "S4": return SCENARIO_CONTROLS["S4"]["ambient_C"]
    if scenario_id == "S5": return SCENARIO_CONTROLS["S5"]["ambient_C"]
    if scenario_id == "S6": return SCENARIO_CONTROLS["S5"]["ambient_C"]
    raise KeyError(scenario_id)

def charge_rate_for_scenario(scenario_id):
    if scenario_id == "S1": return SCENARIO_CONTROLS["S1"]["charge_rate_C"]
    if scenario_id == "S2": return SCENARIO_CONTROLS["S2"]["charge_rate_C"]
    if scenario_id == "S3": return SCENARIO_CONTROLS["S3"]["charge_rate_C"]
    if scenario_id == "S4": return SCENARIO_CONTROLS["S4"]["charge_rate_C"]
    if scenario_id == "S5": return SCENARIO_CONTROLS["S5"]["charge_rate_C"]
    raise KeyError(scenario_id)

print("Trajectory extraction functions loaded.")


In [ ]:
# Cell 20 — Radial particle profile extraction for the PINN training target

def extract_negative_particle_profiles(run, scenario_id, instance_id, cell_id, split, state_kind, fault_cells):
    sol = run["solution"]
    model = run["model"]
    params = run["params"]

    required_profile_variables = {
        "negative_particle_concentration": CORE_VARIABLES["neg_conc"],
        "positive_particle_concentration": CORE_VARIABLES["pos_conc"],
        "negative_radial_stress": CORE_VARIABLES["neg_stress_r"],
        "negative_tangential_stress": CORE_VARIABLES["neg_stress_t"],
    }

    radius = parameter_first(
        params,
        ["Negative particle radius [m]"]
    )
    r_norm = np.linspace(0.0, 1.0, PROFILE_R_POINTS)
    r_m = radius * r_norm

    t_raw = solution_time(sol)
    t_eval = np.linspace(float(t_raw.min()), float(t_raw.max()), PROFILE_T_POINTS)

    neg_thickness = parameter_first(
        params,
        ["Negative electrode thickness [m]"]
    )
    x_pick = 0.5 * neg_thickness

    rows = []

    def normalize_rt(arr, nr, nt):
        arr = np.asarray(arr, dtype=float)
        if arr.shape == (nr, nt):
            return arr
        if arr.shape == (nt, nr):
            return arr.T

        candidates = [i for i, s in enumerate(arr.shape) if s == nr]
        candidates_t = [i for i, s in enumerate(arr.shape) if s == nt]
        if not candidates or not candidates_t:
            raise ValueError(
                f"Cannot orient profile with shape {arr.shape}, nr={nr}, nt={nt}"
            )

        r_axis = candidates[0]
        t_axis = candidates_t[-1]
        arr = np.moveaxis(arr, [r_axis, t_axis], [0, 1])

        if arr.ndim > 2:
            arr = np.nanmean(arr.reshape(nr, nt, -1), axis=2)

        return arr

    for field_name, variable_key in required_profile_variables.items():
        variable = sol[variable_key]

        try:
            if field_name in {"negative_particle_concentration", "positive_particle_concentration"}:
                raw = variable(r=r_m, t=t_eval)
            else:
                raw = variable(r=r_m, t=t_eval, x=np.array([x_pick]))
        except Exception as exc:
            raise RuntimeError(
                f"STOP — could not evaluate required profile '{field_name}' "
                f"using PyBaMM variable '{variable_key}'. Original error: {exc}"
            )

        profile = normalize_rt(raw, len(r_m), len(t_eval))

        if field_name.endswith("stress"):
            profile = profile / 1e6  # Pa -> MPa

        unit = "mol/m3" if "concentration" in field_name else "MPa"

        for j, t_val in enumerate(t_eval):
            for i, r_val in enumerate(r_norm):
                rows.append({
                    "scenario_id": scenario_id,
                    "scenario_instance_id": instance_id,
                    "data_split": split,
                    "pack_topology": PACK_TOPOLOGY,
                    "cell_id": cell_id,
                    "fault_cells_json": json.dumps(fault_cells),
                    "fault_count": len(fault_cells),
                    "state_kind": state_kind,
                    "ageing_cycles": total_aging_cycles if state_kind == "aged" else 0,
                    "time_s": float(t_val),
                    "time_normalized": float(
                        (t_val - t_eval[0]) / max(t_eval[-1] - t_eval[0], 1e-12)
                    ),
                    "r_norm": float(r_val),
                    "r_m": float(r_val * radius),
                    "field": field_name,
                    "unit": unit,
                    "value": float(profile[i, j]),
                })

    return pd.DataFrame(rows)

print("Radial profile extraction function loaded.")


In [ ]:
# Cell 21 — Pack abstraction: 100 independent cells, common current control

# IMPORTANT:
# This is a deliberate 100S1P software abstraction. Each cell is simulated
# with PyBaMM and the same commanded current, then cell voltages are summed.
# It is NOT a coupled electrical pack/network solver and must not be described
# as one later. A future pack-network notebook can replace this abstraction.

def interpolate_to_master(df, master_time, column):
    x = df["time_s"].to_numpy(dtype=float)
    y = df[column].to_numpy(dtype=float)
    order = np.argsort(x)
    x, y = x[order], y[order]

    out = np.full_like(master_time, np.nan, dtype=float)
    inside = (master_time >= x.min()) & (master_time <= x.max())
    if inside.any():
        out[inside] = np.interp(master_time[inside], x, y)
    return out

def assemble_pack_timeseries(cell_frames, scenario_id, instance_id, split, fault_cells):
    max_common_time = min(frame["time_s"].max() for frame in cell_frames)
    master_time = np.linspace(
        0.0,
        float(max_common_time),
        max(51, PROFILE_T_POINTS * 3),
    )

    merged = []
    voltage_stack = []

    for cell_df in cell_frames:
        cell_id = int(cell_df["cell_id"].iloc[0])
        v = interpolate_to_master(cell_df, master_time, "cell_voltage_V")
        current_vals = interpolate_to_master(cell_df, master_time, "cell_current_A")
        voltage_stack.append(v)

        row = pd.DataFrame({
            "scenario_id": scenario_id,
            "scenario_instance_id": instance_id,
            "data_split": split,
            "pack_topology": PACK_TOPOLOGY,
            "cell_id": cell_id,
            "fault_cells_json": json.dumps(fault_cells),
            "fault_count": len(fault_cells),
            "time_s": master_time,
            "pack_voltage_V": np.nan,
            "pack_current_A": current_vals,
            "cell_voltage_V_interp": v,
        })

        merged.append(row)

    voltage_stack = np.vstack(voltage_stack)
    pack_voltage = np.nansum(voltage_stack, axis=0)

    result = pd.concat(merged, ignore_index=True)
    result["pack_voltage_V"] = np.tile(pack_voltage, len(merged))

    return result

print("100-cell pack abstraction helper loaded.")


In [ ]:
# Cell 22 — Scenario instance registry

BASE_INSTANCES = [
    {"instance_id": "S1_HEALTHY", "scenario_id": "S1", "fault_cells": [], "split": "train"},
    {"instance_id": "S2_HEALTHY", "scenario_id": "S2", "fault_cells": [], "split": "train"},
    {"instance_id": "S3_HEALTHY", "scenario_id": "S3", "fault_cells": [], "split": "val"},
]

SCENARIO_INSTANCES = (
    BASE_INSTANCES +
    TRAIN_FAULT_INSTANCES +
    VAL_FAULT_INSTANCES +
    TEST_FAULT_INSTANCES
)

seen = set()
for item in SCENARIO_INSTANCES:
    if item["instance_id"] in seen:
        raise ValueError(f"Duplicate scenario instance: {item['instance_id']}")
    seen.add(item["instance_id"])

    if item["scenario_id"] not in {"S1", "S2", "S3", "S4", "S5"}:
        raise ValueError(f"Unsupported scenario ID: {item['scenario_id']}")

    if not all(1 <= int(c) <= N_CELLS for c in item["fault_cells"]):
        raise ValueError(f"Fault cell outside 1..{N_CELLS}: {item}")

SCENARIO_INSTANCES_DF = pd.DataFrame(SCENARIO_INSTANCES)

print(SCENARIO_INSTANCES_DF.to_string(index=False))


In [ ]:
# Cell 23 — Run healthy reference trajectories once for S1-S5

healthy_runs = {}
for scenario_id in ["S1", "S2", "S3", "S4", "S5"]:
    ctrl = SCENARIO_CONTROLS[scenario_id]
    print(f"Running healthy {scenario_id}: {ctrl['name']}")

    run = run_operating_scenario(
        state_kind="healthy",
        charge_rate_C=ctrl["charge_rate_C"],
        ambient_C=ctrl["ambient_C"],
        healthy_start_solution=healthy_start_solution,
        aged_start_solution=aged_start_solution,
        healthy_capacity_Ah=healthy_capacity_Ah,
        aged_capacity_Ah=aged_capacity_Ah,
    )
    healthy_runs[scenario_id] = run

print("Healthy S1-S5 trajectories generated.")


In [ ]:
# Cell 25 — Expand each scenario instance to all 100 cells and collect scalar/profile data

cell_timeseries_parts = []
profile_parts = []
pack_parts = []

for item in SCENARIO_INSTANCES:
    sid = item["scenario_id"]
    iid = item["instance_id"]
    split = item["split"]
    fault_cells = [int(x) for x in item["fault_cells"]]

    if sid in {"S1", "S2", "S3"}:
        healthy_run = healthy_runs[sid]
        aged_run = None
    else:
        healthy_run = run_operating_scenario(
            state_kind="healthy",
            charge_rate_C=SCENARIO_CONTROLS[sid]["charge_rate_C"],
            ambient_C=SCENARIO_CONTROLS[sid]["ambient_C"],
            healthy_start_solution=healthy_start_solution,
            aged_start_solution=aged_start_solution,
            healthy_capacity_Ah=healthy_capacity_Ah,
            aged_capacity_Ah=aged_capacity_Ah,
        )
        aged_run = aged_runs[sid]

    cell_frames = []

    for cell_id in range(1, N_CELLS + 1):
        is_fault = cell_id in fault_cells
        state = "aged" if is_fault else "healthy"
        run = aged_run if is_fault else healthy_run

        ts = extract_timeseries(
            run,
            sid,
            iid,
            cell_id,
            split,
            state,
            fault_cells,
        )

        prof = extract_negative_particle_profiles(
            run,
            sid,
            iid,
            cell_id,
            split,
            state,
            fault_cells,
        )

        cell_timeseries_parts.append(ts)
        profile_parts.append(prof)
        cell_frames.append(ts)

    pack_df = assemble_pack_timeseries(
        cell_frames,
        scenario_id=sid,
        instance_id=iid,
        split=split,
        fault_cells=fault_cells,
    )
    pack_parts.append(pack_df)

    print(f"{iid}: generated {N_CELLS} cell trajectories.")

cell_timeseries = pd.concat(cell_timeseries_parts, ignore_index=True)
particle_profiles = pd.concat(profile_parts, ignore_index=True)
pack_timeseries = pd.concat(pack_parts, ignore_index=True)

print("\nGenerated raw tables:")
print("cell_timeseries:", cell_timeseries.shape)
print("particle_profiles:", particle_profiles.shape)
print("pack_timeseries:", pack_timeseries.shape)


In [ ]:
# Cell 25 — Expand every declared scenario instance to all 100 cells

cell_timeseries_parts = []
profile_parts = []
pack_parts = []

for item in SCENARIO_INSTANCES:
    sid = item["scenario_id"]
    iid = item["instance_id"]
    split = item["split"]
    fault_cells = [int(x) for x in item["fault_cells"]]

    healthy_run = healthy_runs[sid]
    aged_run = aged_runs[sid] if sid in aged_runs else None

    cell_frames = []

    for cell_id in range(1, N_CELLS + 1):
        is_fault = cell_id in fault_cells
        state = "aged" if is_fault else "healthy"

        if state == "healthy":
            run = healthy_run
        else:
            run = aged_run
            if run is None:
                raise RuntimeError(f"STOP — no aged run available for {sid}")

        ts = extract_timeseries(
            run,
            sid,
            iid,
            cell_id,
            split,
            state,
            fault_cells,
        )

        prof = extract_negative_particle_profiles(
            run,
            sid,
            iid,
            cell_id,
            split,
            state,
            fault_cells,
        )

        cell_timeseries_parts.append(ts)
        profile_parts.append(prof)
        cell_frames.append(ts)

    pack_df = assemble_pack_timeseries(
        cell_frames,
        scenario_id=sid,
        instance_id=iid,
        split=split,
        fault_cells=fault_cells,
    )
    pack_parts.append(pack_df)

    print(f"{iid}: generated {N_CELLS} cell trajectories.")

cell_timeseries = pd.concat(cell_timeseries_parts, ignore_index=True)
particle_profiles = pd.concat(profile_parts, ignore_index=True)
pack_timeseries = pd.concat(pack_parts, ignore_index=True)

print("\nGenerated raw tables:")
print("cell_timeseries:", cell_timeseries.shape)
print("particle_profiles:", particle_profiles.shape)
print("pack_timeseries:", pack_timeseries.shape)


In [ ]:
# Cell 26 — Generate S6 intervention / what-if candidates from the same degraded S5 state

s6_rows = []
s6_cell_rows = []
s6_pack_parts = []

# S6 is an intervention dataset. We do not assign a winner.
for rate in S6_RATES_C:
    iid = f"S6_RATE_{rate:g}C"
    print(f"S6 candidate: {rate:g}C")

    aged_run = run_operating_scenario(
        state_kind="aged",
        charge_rate_C=rate,
        ambient_C=SCENARIO_CONTROLS["S5"]["ambient_C"],
        healthy_start_solution=healthy_start_solution,
        aged_start_solution=aged_start_solution,
        healthy_capacity_Ah=healthy_capacity_Ah,
        aged_capacity_Ah=aged_capacity_Ah,
    )

    healthy_run = healthy_runs["S5"]
    cell_frames = []

    # Candidate-level summary is deliberately descriptive only.
    aged_ts_for_summary = extract_timeseries(
        aged_run,
        scenario_id="S6",
        instance_id=iid,
        cell_id=37,
        split="test",
        state_kind="aged",
        fault_cells=[37],
        ambient_C_override=SCENARIO_CONTROLS["S5"]["ambient_C"],
        charge_rate_C_override=rate,
    )

    max_temp_C = float(np.nanmax(aged_ts_for_summary["cell_temperature_K"] - 273.15))
    min_voltage_V = float(np.nanmin(aged_ts_for_summary["cell_voltage_V"]))
    max_abs_current_A = float(np.nanmax(np.abs(aged_ts_for_summary["cell_current_A"])))

    s6_rows.append({
        "scenario_id": "S6",
        "scenario_instance_id": iid,
        "base_scenario": "S5",
        "candidate_charge_rate_C": rate,
        "fault_cells_json": json.dumps([37]),
        "cells_scanned": N_CELLS,
        "candidate_status": "SIMULATED",
        "max_cell_temperature_C": max_temp_C,
        "min_cell_voltage_V": min_voltage_V,
        "max_abs_current_A": max_abs_current_A,
        "winner_preselected": False,
        "decision_status": "UNRESOLVED_UNTIL_VALIDATED_CONSTRAINTS",
    })

    for cell_id in range(1, N_CELLS + 1):
        is_fault = cell_id == 37
        cell_run = aged_run if is_fault else healthy_run

        cts = extract_timeseries(
            cell_run,
            scenario_id="S6",
            instance_id=iid,
            cell_id=cell_id,
            split="test",
            state_kind="aged" if is_fault else "healthy",
            fault_cells=[37],
            ambient_C_override=SCENARIO_CONTROLS["S5"]["ambient_C"],
            charge_rate_C_override=rate,
        )
        s6_cell_rows.append(cts)
        cell_frames.append(cts)

    s6_pack_parts.append(
        assemble_pack_timeseries(
            cell_frames,
            scenario_id="S6",
            instance_id=iid,
            split="test",
            fault_cells=[37],
        )
    )

S6_SUMMARY = pd.DataFrame(s6_rows)
S6_CELL_TIMESERIES = pd.concat(s6_cell_rows, ignore_index=True)
S6_PACK_TIMESERIES = pd.concat(s6_pack_parts, ignore_index=True)
pack_timeseries = pd.concat([pack_timeseries, S6_PACK_TIMESERIES], ignore_index=True)

assert sorted(S6_SUMMARY["candidate_charge_rate_C"].tolist()) == sorted(S6_RATES_C)
assert (S6_SUMMARY["cells_scanned"] == N_CELLS).all()
assert not S6_SUMMARY["winner_preselected"].any()

print(S6_SUMMARY.to_string(index=False))


In [ ]:
# Cell 28 — Build explicit scenario-instance ground truth records

scenario_records = []

for item in SCENARIO_INSTANCES:
    sid = item["scenario_id"]
    fault_cells = [int(x) for x in item["fault_cells"]]

    scenario_records.append({
        "scenario_instance_id": item["instance_id"],
        "scenario_id": sid,
        "split": item["split"],
        "pack_topology": PACK_TOPOLOGY,
        "cell_count": N_CELLS,
        "fault_cells_json": json.dumps(fault_cells),
        "fault_count": len(fault_cells),
        "fault_location_is_hidden_to_model": True,
        "cell37_is_benchmark": (fault_cells == [37]),
        "healthy_reference_required": True,
        "degradation_ground_truth_source": "PyBaMM_OKane2022_coupled_ageing",
        "degradation_parameter_multipliers_used": False,
        "threshold_status": "UNRESOLVED_UNTIL_VALIDATED",
        "localization_status": "GROUND_TRUTH_AVAILABLE_FOR_EVALUATION_ONLY",
    })

for _, row in S6_SUMMARY.iterrows():
    scenario_records.append({
        "scenario_instance_id": row["scenario_instance_id"],
        "scenario_id": "S6",
        "split": "test",
        "pack_topology": PACK_TOPOLOGY,
        "cell_count": N_CELLS,
        "fault_cells_json": row["fault_cells_json"],
        "fault_count": 1,
        "fault_location_is_hidden_to_model": True,
        "cell37_is_benchmark": True,
        "healthy_reference_required": True,
        "degradation_ground_truth_source": "PyBaMM_OKane2022_coupled_ageing",
        "degradation_parameter_multipliers_used": False,
        "candidate_charge_rate_C": row["candidate_charge_rate_C"],
        "winner_preselected": False,
        "threshold_status": "UNRESOLVED_UNTIL_VALIDATED",
        "localization_status": "GROUND_TRUTH_AVAILABLE_FOR_EVALUATION_ONLY",
    })

scenario_instances_final = pd.DataFrame(scenario_records)

print("Scenario-instance records:", scenario_instances_final.shape)
print(scenario_instances_final.head(10).to_string(index=False))


In [ ]:
# Cell 29 — Dataset sanity checks before writing files

def require_finite(df, columns, name):
    for col in columns:
        vals = pd.to_numeric(df[col], errors="coerce").to_numpy()
        if not np.isfinite(vals).all():
            raise ValueError(f"STOP — non-finite values in {name}.{col}")

required_ts_columns = [
    "scenario_id", "scenario_instance_id", "data_split", "cell_id",
    "time_s", "cell_voltage_V", "cell_current_A",
    "cell_temperature_K", "surface_temperature_K", "soc_proxy",
    "negative_particle_concentration_mean_mol_m3",
    "positive_particle_concentration_mean_mol_m3",
    "electrolyte_concentration_xavg_mol_m3",
    "capacity_Ah_reference_test", "capacity_retention_soh",
]

required_profile_columns = [
    "scenario_id", "scenario_instance_id", "cell_id",
    "time_s", "r_norm", "field", "value"
]

for col in required_ts_columns:
    if col not in cell_timeseries.columns:
        raise KeyError(f"STOP — missing required time-series column: {col}")

for col in required_profile_columns:
    if col not in particle_profiles.columns:
        raise KeyError(f"STOP — missing required profile column: {col}")

require_finite(
    cell_timeseries,
    [
        "cell_voltage_V", "cell_current_A",
        "cell_temperature_K", "surface_temperature_K",
        "soc_proxy", "negative_particle_concentration_mean_mol_m3",
        "positive_particle_concentration_mean_mol_m3",
        "electrolyte_concentration_xavg_mol_m3",
    ],
    "cell_timeseries",
)

require_finite(
    particle_profiles,
    ["time_s", "r_norm", "r_m", "value"],
    "particle_profiles",
)

if not cell_timeseries["cell_id"].between(1, N_CELLS).all():
    raise ValueError("STOP — cell_id outside 1..100.")

if set(cell_timeseries["data_split"].unique()) - {"train", "val", "test"}:
    raise ValueError("STOP — invalid data split label.")

if not (particle_profiles["r_norm"].between(0, 1).all()):
    raise ValueError("STOP — radial coordinate outside [0,1].")

if particle_profiles["field"].nunique() < 3:
    raise ValueError("STOP — fewer than 3 required particle profile fields were generated.")

print("Dataset validation gates: PASS")
print("Cell rows:", len(cell_timeseries))
print("Profile rows:", len(particle_profiles))
print("Scenario instances:", len(scenario_instances_final))


In [ ]:
# Cell 30 — Cross-scenario and localization integrity checks

expected_cells = set(range(1, N_CELLS + 1))

for iid, g in cell_timeseries.groupby("scenario_instance_id"):
    found = set(g["cell_id"].astype(int).unique())
    if found != expected_cells:
        raise ValueError(
            f"STOP — {iid} does not contain all 100 cells. "
            f"Missing={sorted(expected_cells-found)[:10]}, extra={sorted(found-expected_cells)[:10]}"
        )

# The model-facing dataset contains no explicit fault-cell input feature.
for forbidden in ["fault_cell_id", "defective_cell_id", "known_fault_location"]:
    if forbidden in cell_timeseries.columns:
        raise ValueError(f"STOP — forbidden location leakage column found: {forbidden}")

# Confirm the controlled benchmark exists.
bench = scenario_instances_final[
    (scenario_instances_final["scenario_id"].isin(["S4", "S5"])) &
    (scenario_instances_final["cell37_is_benchmark"] == True)
]
if len(bench) == 0:
    raise ValueError("STOP — Cell 37 controlled benchmark is missing.")

# Confirm hidden locations are not all Cell 37.
hidden_fault_cells = set()
for x in scenario_instances_final["fault_cells_json"]:
    hidden_fault_cells.update(json.loads(x))
if hidden_fault_cells <= {37}:
    raise ValueError("STOP — dataset would allow a Cell-37-only shortcut.")

print("Whole-pack / leakage / benchmark integrity: PASS")
print("Unique fault locations represented:", sorted(hidden_fault_cells))


In [ ]:
# Cell 31 — S1 healthy reference vs aged benchmark scientific separation

healthy_s1 = cell_timeseries[
    (cell_timeseries["scenario_instance_id"] == "S1_HEALTHY")
]
aged_s4_instances = scenario_instances_final[
    (scenario_instances_final["scenario_id"] == "S4") &
    (scenario_instances_final["fault_count"] > 0)
]["scenario_instance_id"].tolist()

aged_s4 = cell_timeseries[
    cell_timeseries["scenario_instance_id"].isin(aged_s4_instances) &
    (cell_timeseries["state_kind"] == "aged")
]

healthy_soh_values = healthy_s1["capacity_retention_soh"].dropna().unique()
aged_soh_values = aged_s4["capacity_retention_soh"].dropna().unique()

print("Healthy SOH values:", healthy_soh_values[:5])
print("Aged SOH values:", aged_soh_values[:5])

if len(aged_soh_values) == 0:
    raise ValueError("STOP — no aged-cell SOH state is present.")

if np.nanmin(aged_soh_values) >= 1.0:
    raise ValueError("STOP — aged state did not change capacity retention.")

print("Healthy vs aged state separation: PASS")


In [ ]:
# Cell 32 — Generate research-facing summary plots (not a success claim)

# Plot 1: healthy vs degraded temperature for the benchmark scenario.
plt.figure(figsize=(9, 5))

h = cell_timeseries[
    (cell_timeseries["scenario_instance_id"] == "S4_CELL037_BENCH") &
    (cell_timeseries["state_kind"] == "healthy") &
    (cell_timeseries["cell_id"] == 37)
]
d = cell_timeseries[
    (cell_timeseries["scenario_instance_id"] == "S4_CELL037_BENCH") &
    (cell_timeseries["state_kind"] == "aged") &
    (cell_timeseries["cell_id"] == 37)
]

if len(h) and len(d):
    plt.plot(h["time_s"], h["cell_temperature_K"] - 273.15, label="Healthy Cell 37")
    plt.plot(d["time_s"], d["cell_temperature_K"] - 273.15, label="Degraded Cell 37")

plt.xlabel("Time [s]")
plt.ylabel("Cell temperature [°C]")
plt.title("Controlled benchmark: healthy vs degraded Cell 37")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

# Plot 2: S5 pack voltage for one test instance.
plt.figure(figsize=(9, 5))
p = pack_timeseries[
    pack_timeseries["scenario_instance_id"] == "S5_CELL004_082"
]
if len(p):
    one = p[p["cell_id"] == 1]
    plt.plot(one["time_s"], one["pack_voltage_V"])
plt.xlabel("Time [s]")
plt.ylabel("Pack voltage [V]")
plt.title("S5 pack-voltage trajectory (100S1P abstraction)")
plt.grid(True, alpha=0.25)
plt.show()

print("Plots generated for inspection only.")


In [ ]:
# Cell 32 — Research-facing diagnostic plots (inspection only; no success claim)

# Plot 1: healthy vs degraded Cell 37 under matched S4 controls.
plt.figure(figsize=(9, 5))

h = cell_timeseries[
    (cell_timeseries["scenario_instance_id"] == "S4_CELL037_BENCH") &
    (cell_timeseries["state_kind"] == "healthy") &
    (cell_timeseries["cell_id"] == 37)
]
d = cell_timeseries[
    (cell_timeseries["scenario_instance_id"] == "S4_CELL037_BENCH") &
    (cell_timeseries["state_kind"] == "aged") &
    (cell_timeseries["cell_id"] == 37)
]

if len(h):
    plt.plot(h["time_s"], h["cell_temperature_K"] - 273.15, label="Healthy Cell 37")
if len(d):
    plt.plot(d["time_s"], d["cell_temperature_K"] - 273.15, label="Degraded Cell 37")

plt.xlabel("Time [s]")
plt.ylabel("Cell temperature [°C]")
plt.title("Controlled benchmark: healthy vs degraded Cell 37")
if len(h) or len(d):
    plt.legend()
plt.grid(True, alpha=0.25)
plt.show()

# Plot 2: first available S5 pack instance.
plt.figure(figsize=(9, 5))

s5_available = pack_timeseries.loc[
    pack_timeseries["scenario_id"].eq("S5"),
    "scenario_instance_id"
].drop_duplicates().tolist()

if s5_available:
    iid = s5_available[0]
    one = pack_timeseries[
        (pack_timeseries["scenario_instance_id"] == iid) &
        (pack_timeseries["cell_id"] == 1)
    ]
    plt.plot(one["time_s"], one["pack_voltage_V"])
    plt.title(f"{iid}: pack-voltage trajectory (100S1P abstraction)")
else:
    plt.title("No S5 pack instance in SMOKE mode")

plt.xlabel("Time [s]")
plt.ylabel("Pack voltage [V]")
plt.grid(True, alpha=0.25)
plt.show()

print("Plots generated for inspection only.")


In [ ]:
# Cell 34 — Final artifact hashes + validation report

artifacts = [
    TS_PATH,
    PROFILE_PATH,
    PACK_PATH,
    SCENARIO_PATH,
    S6_PATH,
    V1_DIR / "v1_data_dictionary.json",
    V1_DIR / "v1_generation_metadata.json",
]

artifact_hashes = {}
for path in artifacts:
    if not path.is_file() or path.stat().st_size == 0:
        raise IOError(f"STOP — expected artifact missing/empty: {path}")
    artifact_hashes[path.name] = sha256_file(path)

validation_report = {
    "status": "PASS",
    "generated_at_utc": timestamp,
    "run_mode": RUN_MODE,
    "checks": {
        "v0_lineage_verified": True,
        "notebook_03_contract_verified": True,
        "pybamm_smoke_solve": True,
        "healthy_reference_generated": True,
        "physics_based_ageing_generated": True,
        "100_cells_present_per_instance": True,
        "fault_location_leakage_absent": True,
        "cell37_benchmark_present": True,
        "non37_fault_locations_present": True,
        "s6_candidate_count": len(S6_RATES_C),
        "s6_winner_preselected": False,
        "risk_thresholds_declared_validated": False,
    },
    "dataset_shapes": {
        "cell_timeseries_rows": int(len(cell_timeseries)),
        "particle_profile_rows": int(len(particle_profiles)),
        "pack_timeseries_rows": int(len(pack_timeseries)),
        "scenario_instances": int(len(scenario_instances_final)),
        "s6_candidates": int(len(S6_SUMMARY)),
    },
    "artifact_sha256": artifact_hashes,
}

with open(V1_DIR / "v1_generation_validation.json", "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2)

with open(V1_DIR / "v1_artifact_hash_manifest.json", "w", encoding="utf-8") as f:
    json.dump(artifact_hashes, f, indent=2)

print("=" * 88)
print("V1 PYBAMM DATA GENERATOR — FINAL STATUS")
print("=" * 88)
print("STATUS: PASS")
print("Run mode:", RUN_MODE)
print("Cell time-series rows:", len(cell_timeseries))
print("Particle profile rows:", len(particle_profiles))
print("Pack time-series rows:", len(pack_timeseries))
print("Scenario instances:", len(scenario_instances_final))
print("S6 candidates:", len(S6_SUMMARY))
print("Winner preselected:", False)
print("Risk thresholds validated:", False)
print("=" * 88)


In [ ]:
# Cell 35 — Write the downstream handoff note for Notebook 05

next_action = {
    "current_notebook": "04_V1_PyBaMM_DataGenerator_FINAL.ipynb",
    "status": "COMPLETED",
    "next_notebook": "05_V1_DatasetValidation_GroundTruth.ipynb",
    "required_next_work": [
        "Validate feature distributions and units.",
        "Check scenario balance and split integrity.",
        "Quantify healthy-vs-aged separability.",
        "Verify that the known fault location is used only as evaluation ground truth.",
        "Define future-horizon target construction without leakage.",
        "Prepare external experimental-data compatibility checks.",
        "Do not assign operational safety thresholds until sourced and validated.",
    ],
    "important_boundary": (
        "The dataset is PyBaMM simulation-generated reference data, not experimental hardware data. "
        "A future external-data validation stage is required before experimental claims."
    ),
}

with open(V1_DIR / "NEXT_ACTION_NOTEBOOK_05.json", "w", encoding="utf-8") as f:
    json.dump(next_action, f, indent=2)

print(next_action)


## Stop here

The next scientific stage is **Notebook 05 — Dataset Validation / Ground Truth**.

Do not train the V1 PINN immediately after Notebook 04. First verify that the generated dataset is physically coherent, split-safe, leakage-free, and contains a measurable but explainable healthy/degraded separation.

### Important interpretation
This notebook deliberately improves the old V1 specification in one place:

- Notebook 03 contained **provisional** degradation multipliers for future mapping.
- Notebook 04 does **not** use those multipliers as physical ground truth.
- Instead, the degraded state is generated by the coupled PyBaMM ageing model using the declared `OKane2022` parameter set and degradation mechanisms.

That means the next notebook must validate the resulting state rather than assuming that the degradation is correct.
